# Run 3 — Embeddings statiques vs dynamiques
## Word2Vec + SVM vs CamemBERT + SVM




In [1]:
# ── Installation des librairies ──────────────────────────────────
!pip install gensim transformers torch scikit-learn pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 61.6 MB/s eta 0:00:00


In [2]:
# ── Imports ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import torch
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from transformers import AutoTokenizer, AutoModel
from sklearn.svm import SVC
from sklearn.metrics import f1_score, classification_report
from google.colab import files

print('✅ Imports OK')
print(f'GPU disponible : {torch.cuda.is_available()}')

✅ Imports OK
GPU disponible : True


In [4]:
# ── Chargement des données ───────────────────────────────────────
# Sélectionne train.csv ET test.csv quand la fenêtre s'ouvre

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

y_train = train['type']
y_test  = test['type']

print(f'Train : {len(train)} recettes')
print(f'Test  : {len(test)} recettes')
print(f'Classes : {train["type"].value_counts().to_dict()}')

Train : 12473 recettes
Test  : 1388 recettes
Classes : {'Plat principal': 5802, 'Dessert': 3762, 'Entrée': 2909}


In [5]:
# ── Preprocessing ────────────────────────────────────────────────
def preprocess(text):
    text = re.sub(r'[^\w\s]', ' ', str(text))
    text = re.sub(r'\d+', '', text)
    return text.lower()

# Concaténer titre + ingredients + recette
train['texte'] = (train['titre'] + ' ' + train['ingredients'] + ' ' + train['recette']).apply(preprocess)
test['texte']  = (test['titre']  + ' ' + test['ingredients']  + ' ' + test['recette']).apply(preprocess)

print('✅ Preprocessing OK')

✅ Preprocessing OK


---
## Run 3a — Word2Vec + SVM (Embeddings statiques)

Chaque mot a **un seul vecteur fixe** quel que soit son contexte.  
La recette est représentée par la **moyenne** des vecteurs de ses tokens.

In [6]:
# ── Entraîner Word2Vec sur le corpus train ───────────────────────
print('Entraînement Word2Vec...')
sentences = [simple_preprocess(t) for t in train['texte']]

w2v = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=10
)
print(f'✅ Word2Vec entraîné — vocab : {len(w2v.wv)} mots')

Entraînement Word2Vec...
✅ Word2Vec entraîné — vocab : 11866 mots


In [7]:
# ── Vectorisation : moyenne des vecteurs Word2Vec ────────────────
def doc_vector(text, model):
    tokens = simple_preprocess(text)
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

print('Vectorisation train...')
X_train_w2v = np.array([doc_vector(t, w2v) for t in train['texte']])
print('Vectorisation test...')
X_test_w2v  = np.array([doc_vector(t, w2v) for t in test['texte']])
print(f'✅ Shape train : {X_train_w2v.shape}, test : {X_test_w2v.shape}')

Vectorisation train...
Vectorisation test...
✅ Shape train : (12473, 100), test : (1388, 100)


In [8]:
# ── SVM Word2Vec ─────────────────────────────────────────────────
print('Entraînement SVM Word2Vec...')
clf_w2v = SVC(kernel='linear', C=1.0)
clf_w2v.fit(X_train_w2v, y_train)
y_pred_w2v = clf_w2v.predict(X_test_w2v)

f1_w2v = f1_score(y_test, y_pred_w2v, average='macro')

print('\n=== Run 3a — Word2Vec + SVM ===')
print(classification_report(y_test, y_pred_w2v))
print(f'F1 macro Word2Vec : {f1_w2v:.3f}')

# Sauvegarder les prédictions
pd.Series(y_pred_w2v).to_csv('predictions_run3a_word2vec.csv', index=False)

Entraînement SVM Word2Vec...

=== Run 3a — Word2Vec + SVM ===
                precision    recall  f1-score   support

       Dessert       0.98      0.99      0.99       407
        Entrée       0.75      0.64      0.69       337
Plat principal       0.83      0.89      0.86       644

      accuracy                           0.86      1388
     macro avg       0.85      0.84      0.84      1388
  weighted avg       0.85      0.86      0.85      1388

F1 macro Word2Vec : 0.844


---
## Run 3b — CamemBERT + SVM (Embeddings dynamiques)

Le même mot a **un vecteur différent selon son contexte**.  
Modèle pré-entraîné sur 138GB de texte français.

In [9]:
# ── Charger CamemBERT ────────────────────────────────────────────
print('Chargement CamemBERT...')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

tokenizer  = AutoTokenizer.from_pretrained('camembert-base')
model_bert = AutoModel.from_pretrained('camembert-base').to(device)
model_bert.eval()
print('✅ CamemBERT chargé')

Chargement CamemBERT...
Device : cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ CamemBERT chargé


In [ ]:
# ── Extraire les embeddings (moyenne du dernier layer) ───────────
def get_embedding(text, max_length=128):
    inputs = tokenizer(
        str(text),
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
        padding='max_length'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model_bert(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

# Sur le titre uniquement (plus rapide, très discriminant)
print('Extraction embeddings train (peut prendre ~10 min)...')
X_train_bert = np.array([get_embedding(t) for t in train['titre']])
print('Extraction embeddings test...')
X_test_bert  = np.array([get_embedding(t) for t in test['titre']])
print(f' Shape train : {X_train_bert.shape}, test : {X_test_bert.shape}')

Extraction embeddings train (peut prendre ~10 min)...
Extraction embeddings test...
✅ Shape train : (12473, 768), test : (1388, 768)


In [11]:
# ── SVM CamemBERT ────────────────────────────────────────────────
print('Entraînement SVM CamemBERT...')
clf_bert = SVC(kernel='linear', C=1.0)
clf_bert.fit(X_train_bert, y_train)
y_pred_bert = clf_bert.predict(X_test_bert)

f1_bert = f1_score(y_test, y_pred_bert, average='macro')

print('\n=== Run 3b — CamemBERT + SVM ===')
print(classification_report(y_test, y_pred_bert))
print(f'F1 macro CamemBERT : {f1_bert:.3f}')

# Sauvegarder les prédictions
pd.Series(y_pred_bert).to_csv('predictions_run3b_camembert.csv', index=False)

Entraînement SVM CamemBERT...

=== Run 3b — CamemBERT + SVM ===
                precision    recall  f1-score   support

       Dessert       0.83      0.87      0.85       407
        Entrée       0.63      0.43      0.51       337
Plat principal       0.72      0.82      0.77       644

      accuracy                           0.74      1388
     macro avg       0.73      0.70      0.71      1388
  weighted avg       0.73      0.74      0.73      1388

F1 macro CamemBERT : 0.708


---
## Récap comparatif

In [12]:
# ── Récap final ──────────────────────────────────────────────────
print('='*50)
print('RÉCAP — Statiques vs Dynamiques')
print('='*50)
print(f'Word2Vec  (statique)   F1 macro = {f1_w2v:.3f}')
print(f'CamemBERT (dynamique)  F1 macro = {f1_bert:.3f}')
print(f'Meilleur  : {"CamemBERT" if f1_bert > f1_w2v else "Word2Vec"}')

# Télécharger les prédictions
files.download('predictions_run3a_word2vec.csv')
files.download('predictions_run3b_camembert.csv')

RÉCAP — Statiques vs Dynamiques
Word2Vec  (statique)   F1 macro = 0.844
CamemBERT (dynamique)  F1 macro = 0.708
Meilleur  : Word2Vec


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>